# 03. Compressed materialization and index

목표: 02에서 고정한 compressor로 전체 원본 임베딩을 변환하고, PCA 검색 벡터와 PQ code를 별도 DB 테이블에 저장한 뒤 pgvector index를 확인합니다. 성공 기준은 source/model attempt가 각 행과 phase log에 연결되는 것입니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 02의 가장 최근 `completed` attempt만 사용합니다. 중단되면 03 전체를 다시 실행하며 같은 run/model의 기존 행은 건너뜁니다. compressor나 원본 임베딩이 바뀌면 02 또는 01부터 다시 시작하고, protocol/config가 바뀌면 00부터 새 run을 만듭니다.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import RunStore, resolve_active_run

EXECUTE_STAGE = False
RUN_ROOT = PROJECT_ROOT / 'runs'
RUN_DIR = resolve_active_run(RUN_ROOT)


## Plan

- Resolve one completed compressor attempt and load its immutable artifacts.
- Materialize PCA-256 as a searchable pgvector and PQ as non-searchable binary codes.
- Record insert/skip counts plus reconstruction/angular error summaries; create vector indexes.


In [ ]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable.')
    return run, manifest

def latest_completed_attempt(run_dir: Path, phase_name: str) -> int:
    attempts = run_dir / 'phases' / phase_name / 'attempts'
    completed = []
    for path in sorted(attempts.glob('A*/phase_manifest.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        if payload.get('status') == 'completed':
            completed.append(int(payload['attempt']))
    if not completed:
        raise RuntimeError(f'No completed attempt for {phase_name}.')
    return max(completed)

preflight = {'execute_stage': EXECUTE_STAGE, 'run_dir_resolved': str(RUN_DIR),
             'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file())}
preflight


## Execute and record

PCA retrieval vector와 512D reconstructed certificate vector를 혼동하지 마십시오. PQ code는 pgvector index 대상이 아닙니다.


In [ ]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    import numpy as np
    from research.compression import PCACompressor, PQCompressor
    from research.database import VectorRepository, create_database_engine, ensure_vector_indexes, load_database_settings, session_scope
    from research.database.models import Embedding256, Embedding512, EmbeddingPQ

    run, _ = attach_run(RUN_DIR)
    run.verify_inputs()
    run.verify_phase_artifacts('02_compressor_fit')
    compressor_attempt = latest_completed_attempt(RUN_DIR, '02_compressor_fit')
    suffix = f'A{compressor_attempt:03d}'
    artifact_dir = RUN_DIR / 'artifacts' / '02_compressor_fit'
    pca = PCACompressor.load(artifact_dir / f'pca_256_{suffix}.joblib')
    pq = PQCompressor.load(artifact_dir / f'pq_{suffix}.faiss')
    engine = create_database_engine(load_database_settings())
    counts = {'source_vectors': 0, 'pca_inserted': 0, 'pca_skipped': 0, 'pq_inserted': 0, 'pq_skipped': 0}
    with session_scope(engine) as session:
        source_rows = session.query(Embedding512).filter(
            Embedding512.vector_type == 'arcface', Embedding512.run_uid == run.run_id
        ).all()
        if not source_rows:
            raise ValueError('No source embeddings for this run.')
        matrix = np.stack([np.asarray(row.embedding, dtype=np.float32) for row in source_rows])
        pca_profile = pca.transform_profile(matrix)
        pq_profile = pq.transform_profile(matrix)
        pca_existing = {row.image_id for row in session.query(Embedding256).filter(
            Embedding256.vector_type == 'pca_256', Embedding256.run_uid == run.run_id
        ).all()}
        pq_existing = {row.image_id for row in session.query(EmbeddingPQ).filter(
            EmbeddingPQ.vector_type == 'pq_auxiliary', EmbeddingPQ.run_uid == run.run_id
        ).all()}
        repository = VectorRepository(session)
        counts['source_vectors'] = len(source_rows)
        for index, source in enumerate(source_rows):
            if source.image_id in pca_existing:
                counts['pca_skipped'] += 1
            else:
                repository.add_embedding_256(source.image_id, 'pca_256',
                    {'run_id': run.run_id, 'compressor_attempt': suffix, 'source_embedding_id': source.id},
                    pca_profile.vectors[index], log='PCA retrieval vector; certificate uses reconstructed 512D artifact',
                    run_uid=run.run_id)
                counts['pca_inserted'] += 1
            if source.image_id in pq_existing:
                counts['pq_skipped'] += 1
            else:
                repository.add_embedding_pq(source.image_id, 'pq_auxiliary',
                    {'run_id': run.run_id, 'compressor_attempt': suffix, 'source_embedding_id': source.id},
                    pq_profile.codes[index].tobytes(), log='PQ auxiliary code; not pgvector-searchable',
                    run_uid=run.run_id)
                counts['pq_inserted'] += 1
    ensure_vector_indexes(engine)
    with run.phase('03_compressed_materialization_and_index') as phase:
        summary = {
            'compressor_attempt': suffix, 'counts': counts,
            'pca_reconstruction_mse_mean': float(np.mean(pca_profile.reconstruction_error)),
            'pca_angular_error_mean': float(np.mean(pca_profile.angular_error)),
            'pq_reconstruction_mse_mean': float(np.mean(pq_profile.reconstruction_error)),
            'pq_angular_error_mean': float(np.mean(pq_profile.angular_error)),
        }
        source = phase.attempt_dir / f'materialization_summary_A{phase.attempt:03d}.json'
        source.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
        phase.publish_artifact(source)
        phase.record_counts(**counts)
    result = {'status': 'completed', 'run_id': run.run_id, **summary}
result


## Next step

insert/skip 합계가 source vector 수와 맞는지 확인합니다. 불일치나 index 실패가 있으면 03부터 재시작하고, 변환 모델 변경은 02부터 새 attempt로 수행합니다.
